**P2-Customer Order Insights And delivery Tracker**

**Week 3 - Order Analysis with PySpark**

In [1]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("P2_Week3").getOrCreate()

In [4]:
from pyspark.sql import functions as F

In [6]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers.csv


**1. Load order and delivery data into PySpark.**

In [8]:
orders = spark.read.csv("cleaned_orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)

print("Orders:", orders.count(), "rows | Customers:", customers.count(), "rows")
print(orders.columns)
orders.show(5)
customers.show()

Orders: 22 rows | Customers: 8 rows
['order_id', 'customer_id', 'product_name', 'quantity', 'amount', 'order_date', 'expected_date', 'delivery_date', 'status', 'issue', 'delay_days', 'delayed']
+--------+-----------+-----------------+--------+------+-------------------+-------------+-------------------+---------+-------------+----------+-------+
|order_id|customer_id|     product_name|quantity|amount|         order_date|expected_date|      delivery_date|   status|        issue|delay_days|delayed|
+--------+-----------+-----------------+--------+------+-------------------+-------------+-------------------+---------+-------------+----------+-------+
|       1|          1|   Wireless Mouse|       1|   799|2026-09-01 10:15:00|   2026-09-06|2026-09-05 14:30:00|Delivered|         NULL|         0|      0|
|       2|          1|       Laptop Bag|       1|  1499|2026-09-10 09:40:00|   2026-09-15|2026-09-18 16:05:00|  Delayed|Courier Delay|         3|      1|
|       3|          1|        USB Ca

**2. Join orders and customer tables.**

In [9]:
joined_df = orders.join(customers, on="customer_id", how="inner")

joined_df.select("order_id", "customer_name", "region", "product_name",
              "status", "delay_days", "delayed").show(10)

+--------+-------------+------+-----------------+---------+----------+-------+
|order_id|customer_name|region|     product_name|   status|delay_days|delayed|
+--------+-------------+------+-----------------+---------+----------+-------+
|       1|   Arun Kumar| South|   Wireless Mouse|Delivered|         0|      0|
|       2|   Arun Kumar| South|       Laptop Bag|  Delayed|         3|      1|
|       3|   Arun Kumar| South|        USB Cable|  Delayed|        10|      1|
|       4| Priya Sharma| North|Bluetooth Speaker|Delivered|         0|      0|
|       5| Priya Sharma| North|       Phone Case|  Delayed|         4|      1|
|       6|  Karthik Raj| South|         Keyboard|Delivered|         0|      0|
|       7|  Karthik Raj| South|    Monitor Stand|Delivered|         0|      0|
|       8|  Karthik Raj| South|       Power Bank|  Delayed|         4|      1|
|       9|  Sneha Patel|  West|       Headphones|Delivered|         0|      0|
|      10|  Sneha Patel|  West|      Smart Watch|  D

**3. Group by region to count delays.**

In [10]:
by_region = joined_df.groupBy("region").agg(
    F.sum("delayed").alias("delayed_orders"),
    F.count("order_id").alias("total_orders")
)

by_region.orderBy("delayed_orders", ascending=False).show(10)

+------+--------------+------------+
|region|delayed_orders|total_orders|
+------+--------------+------------+
| South|             4|           9|
|  East|             2|           3|
| North|             2|           5|
|  West|             1|           5|
+------+--------------+------------+



**4. Save results to a file.**

In [11]:
import glob

by_region.coalesce(1).write.mode("overwrite").option("header", True).csv("output/delayed_orders_by_region")
by_region.write.mode("overwrite").parquet("output/delayed_orders_by_region_parquet")

csv_file = glob.glob("output/delayed_orders_by_region/part-*.csv")[0]
print(open(csv_file).read())

region,delayed_orders,total_orders
South,4,9
East,2,3
West,1,5
North,2,5

